#**CDMX: Inteligencia de Datos contra Riesgos de Clausura**
Este proyecto automatiza la detección de vulnerabilidades administrativas para negocios en la Ciudad de México. Utilizamos datos históricos del INVEA para calcular el riesgo real y el impacto financiero de una suspensión.

Celda de Código 1: Limpieza y Motor de Riesgo

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pandas as pd
import plotly.express as px

# Carga y Limpieza Automática
df = pd.read_csv('/content/drive/MyDrive/csv/suspensiones_invea_cdmx.csv')
df.columns = df.columns.str.strip().str.upper().str.replace(' ', '_')

# Algoritmo de Costo de Riesgo (UMA 2026 + Lucro Cesante)
COSTO_POR_CLAUSURA = (500 * 117.31) + (15 * 8000)
print(f"⚠️ Impacto económico base por clausura: ${COSTO_POR_CLAUSURA:,.2f} MXN")

⚠️ Impacto económico base por clausura: $178,655.00 MXN


**¿Dónde está el peligro?**

Analizamos la densidad de suspensiones por delegación. Las barras más altas representan zonas con operativos activos donde la vigilancia es crítica.

💻 Celda de Código 2: Gráfica por Delegación

In [17]:
fig_delegacion = px.bar(df['ALCALDÍA'].value_counts().head(8),
             title='<b>Ranking de Riesgo por Delegación</b>',
             labels={'value':'Total de Suspensiones', 'index':'Delegación'},
             color_continuous_scale='Reds', color='value')
fig_delegacion.show()

**Impacto Financiero por Sector**

No es solo una multa, es la viabilidad de tu empresa. Este mapa de calor muestra el capital total que ha salido de cada sector debido a cierres evitables.

💻 Celda de Código 3: Gráfica de Pérdidas Monetarias

In [18]:
import pandas as pd
import plotly.express as px

# 1. Configuración de datos (Asegúrate de tener el DF cargado)
# ... (carga de datos previa) ...

# 2. Definición de Costos
UMA_2026 = 117.31
COSTO_POR_EVENTO = (500 * UMA_2026) + (15 * 8000) # Multa + Lucro Cesante

# 3. Preparación de Tabla de Pérdidas
df_paga = df['GIRO'].value_counts().head(8).reset_index()
df_paga.columns = ['GIRO', 'CONTEO']
df_paga['PERDIDA_TOTAL'] = df_paga['CONTEO'] * COSTO_POR_EVENTO

# --- PASO 4: GRÁFICA DE PÉRDIDAS CON LETRA GRANDE ---
fig_costo = px.bar(
    df_paga,
    x='PERDIDA_TOTAL',
    y='GIRO',
    orientation='h',
    title='<b>💸 IMPACTO ECONÓMICO TOTAL POR SECTOR (MXN)</b>',
    text='PERDIDA_TOTAL', # Ponemos el valor sobre la barra
    color='PERDIDA_TOTAL',
    color_continuous_scale='Reds'
)

# AJUSTES DE VISIBILIDAD (Aquí es donde corregimos el tamaño)
fig_costo.update_traces(
    texttemplate='$%{text:,.2s}', # Formato compacto: $15M, $2.5M
    textposition='outside',
    textfont_size=16, # <-- LETRA GRANDE PARA LOS MONTOS
    cliponaxis=False
)

fig_costo.update_layout(
    xaxis_title="Millones de Pesos (MDP)",
    yaxis_title="Giro del Negocio",
    font=dict(size=14), # <-- Aumenta el tamaño general de toda la gráfica
    title_font_size=22,
    yaxis={'categoryorder':'total ascending'},
    height=600,
    margin=dict(l=50, r=100, t=80, b=50),
    template='plotly_white'
)

fig_costo.show()